# Week 4 — Noise Sensitivity Analysis & Threshold Tuning
## Member 2 — ML Engineer & Member 3 — Context & Integration

## 1. Environment Setup & Imports
## 2. Load & Prepare Data
## 3. Train Final Model
## 4. Precision-Recall Curve (Member 2)
## 5. Threshold Sweep (Member 3)
## 6. Robustness Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_curve, auc, 
                             f1_score, precision_score, recall_score)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
import os

os.makedirs('../outputs', exist_ok=True)
print("All imports successful!")

## 2. Load & Prepare Data

In [ ]:
# Load dataset
df = pd.read_csv('../data/ai4i2020.csv')

# Add external context
np.random.seed(42)
df['ambient_temp_C'] = np.random.normal(loc=28, scale=5, size=len(df))
df['factory_load_pct'] = np.random.uniform(50, 100, size=len(df))
df['humidity_pct'] = np.random.normal(loc=60, scale=10, size=len(df))

# Encode Type
le = LabelEncoder()
df['Type_enc'] = le.fit_transform(df['Type'])

# Define features
ext_features = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
    'Type_enc', 'ambient_temp_C', 'factory_load_pct', 'humidity_pct'
]

X = df[ext_features].copy()
y = df['Machine failure']

# Clean feature names for LightGBM
X.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in X.columns]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 3. Train Final Model

In [ ]:
# Build and train final pipeline
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('lgbm', LGBMClassifier(
        random_state=42, n_jobs=-1, verbose=-1,
        n_estimators=500, learning_rate=0.05,
        num_leaves=31, scale_pos_weight=20
    ))
])

pipeline.fit(X_train, y_train)
print("✅ Final model trained successfully!")

## 4. Precision-Recall Curve (Member 2)

In [ ]:
# Get probability predictions on CLEAN test set
y_proba = pipeline.predict_proba(X_test)[:, 1]

# Plot Precision-Recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
auc_pr = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', linewidth=2,
         label=f'Clean Test Set (AUC-PR = {auc_pr:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Clean Test Set')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('../outputs/pr_curve_clean.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC-PR (Clean): {auc_pr:.4f}")

## Commentary — Precision-Recall Curve

The Precision-Recall curve shows the tradeoff between precision (avoiding false alarms) and recall (catching real failures) at different decision thresholds.

**AUC-PR Score:**
- Higher AUC-PR = better model performance on imbalanced datasets
- AUC-PR is more informative than ROC-AUC for highly imbalanced data like this (3.39% failure rate)
- A random classifier would achieve AUC-PR ≈ 0.034 (equal to failure rate)
- Our model significantly outperforms random — confirming LightGBM + SMOTE is effective

**For industrial maintenance deployment:**
- High recall is preferred (catch as many failures as possible)
- Some false alarms are acceptable (cheaper than missed failures)
- Optimal threshold will be tuned in the next section